# 用项目代码加载真实模型，遍历 decoder layer 打印 get_named_linears

**全部调用项目自己的代码**，不手搭玩具：
- 用 `AutoQuantForCausalLM.from_pretrained` 加载真实模型（[quant/core/api.py](quant/core/api.py)）
- 用项目的 `get_named_linears`（[quant/utils/common_utils.py:55](quant/utils/common_utils.py#L55)）

模型选 `facebook/opt-125m`（约 250MB），是项目支持列表里最小的，首次运行会自动下载。

> **Kernel**：用 `micro_env`。依赖已装：`transformers==4.51.3`、`accelerate`、`datasets==3.6.0`（torch 2.2.2）。

## 1. import 项目代码

关掉一堆无关 warning，让 notebook 根目录能找到 `quant` 包。

In [1]:
import warnings
warnings.filterwarnings("ignore")

from quant.core.api import AutoQuantForCausalLM      # 项目的模型加载入口
from quant.utils.common_utils import get_named_linears  # 项目的工具函数

import inspect
print("get_named_linears 的真身：")
print(inspect.getsource(get_named_linears))

get_named_linears 的真身：
def get_named_linears(module):
    return {name: m for name, m in module.named_modules() if isinstance(m, nn.Linear)}



## 2. 用项目代码加载真实模型

`safetensors=False`：opt-125m 这个仓库只有 `pytorch_model.bin`，没有 `.safetensors`，
不加这个参数会报 `no file named model.safetensors`。

返回的 `wrapper` 是项目的 `OptModelForCausalLM`，真正的 HF 模型在 `wrapper.model`。

In [2]:
wrapper = AutoQuantForCausalLM.from_pretrained("facebook/opt-125m", safetensors=False)

print("wrapper 类型 :", type(wrapper).__name__)   # OptModelForCausalLM（项目的包装类）
print("里面的 model :", type(wrapper.model).__name__)  # OPTForCausalLM（HF 原生模型）

model type is  opt


Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 98922.26it/s]


target cls name is  AutoModelForCausalLM
wrapper 类型 : OptModelForCausalLM
里面的 model : OPTForCausalLM


## 3. 拿到所有 decoder layer

用项目的 `get_model_layers`（对 OPT 就是 `model.model.decoder.layers`）。
这一串就相当于 `AwqQuantizer` 里的 `self.target_modules`。

In [3]:
target_modules = wrapper.get_model_layers(wrapper.model)   # 对应 self.target_modules
print("decoder layer 层数 =", len(target_modules))
print("每一层的类型 =", type(target_modules[0]).__name__)

decoder layer 层数 = 12
每一层的类型 = OPTDecoderLayer


## 4. 遍历每一层，打印 get_named_linears

这就是 [quantizer.py:174](quant/quantization/awq/quantizer.py#L174) 那行 `get_named_linears(self.target_modules[i])` 在真实模型上跑出来的结果。

In [4]:
for i in range(len(target_modules)):
    named_linears = get_named_linears(target_modules[i])
    print(f"====== decoder layer {i} ======")
    for name, m in named_linears.items():
        print(f"  {name:18s} -> in={m.in_features:<5d} out={m.out_features:<5d} bias={m.bias is not None}")
    print(f"  （共 {len(named_linears)} 个 Linear）\n")

====== decoder layer 0 ======
  self_attn.k_proj   -> in=768   out=768   bias=True
  self_attn.v_proj   -> in=768   out=768   bias=True
  self_attn.q_proj   -> in=768   out=768   bias=True
  self_attn.out_proj -> in=768   out=768   bias=True
  fc1                -> in=768   out=3072  bias=True
  fc2                -> in=3072  out=768   bias=True
  （共 6 个 Linear）

====== decoder layer 1 ======
  self_attn.k_proj   -> in=768   out=768   bias=True
  self_attn.v_proj   -> in=768   out=768   bias=True
  self_attn.q_proj   -> in=768   out=768   bias=True
  self_attn.out_proj -> in=768   out=768   bias=True
  fc1                -> in=768   out=3072  bias=True
  fc2                -> in=3072  out=768   bias=True
  （共 6 个 Linear）

====== decoder layer 2 ======
  self_attn.k_proj   -> in=768   out=768   bias=True
  self_attn.v_proj   -> in=768   out=768   bias=True
  self_attn.q_proj   -> in=768   out=768   bias=True
  self_attn.out_proj -> in=768   out=768   bias=True
  fc1                -> in

## 5. 说明：OPT 的命名和 Qwen 不一样

你在 [quantizer.py:167](quant/quantization/awq/quantizer.py#L167) 注释里看到的是 **Qwen** 的命名
（`self_attn.q_proj`、`mlp.gate_proj` …）。**OPT 结构相同但名字不同**：

| 作用 | Qwen 里的名字 | OPT 里的名字 |
|---|---|---|
| 注意力 Q/K/V | `self_attn.q/k/v_proj` | `self_attn.q/k/v_proj` |
| 注意力输出 | `self_attn.o_proj` | `self_attn.out_proj` |
| 前馈第一层 | `mlp.gate_proj` / `mlp.up_proj` | `fc1` |
| 前馈第二层 | `mlp.down_proj` | `fc2` |

> 所以每个模型才需要各自的 `get_layers_for_scaling`（[qwen3.py:32](quant/nn_models/models/qwen3.py#L32)、`opt.py`）——
> 用来告诉 AWQ「这个模型里，哪个 Linear 叫什么、谁接在谁后面」。

验证一下 `get_named_linears` 只挑 Linear、把 LayerNorm 等过滤掉：

In [ ]:
import torch.nn as nn
layer0 = target_modules[0]
print("named_modules() 看到的全部子模块（get_named_linears 只保留 ✅ 那些）：")
for name, m in layer0.named_modules():
    tag = "✅ Linear" if isinstance(m, nn.Linear) else "❌ 过滤"
    print(f"  {tag}  {name or '<自己>':22s} {type(m).__name__}")

In [5]:
class Dog:
    species = "Canis"

    @staticmethod
    def static_print_species():
        # 手里没有 cls，只能硬把 "Dog" 的名字写死在这里
        print(f"Species is: {Dog.species}")

    @classmethod
    def class_print_species(cls):
        # 动态接收调用者
        print(f"Species is: {cls.species}")

class Wolf(Dog):
    species = "Lupus" # 子类重写了类属性

# 测试类方法
Wolf.class_print_species() 
# 输出: Species is: Lupus (✅ 完美，cls 动态变成了 Wolf)

# 测试静态方法
Wolf.static_print_species() 
# 输出: Species is: Canis (❌ 出错了！因为方法内部写死了 Dog.species)

Species is: Lupus
Species is: Canis


In [ ]:
print(5 or 6) # 5
print(5 | 6)  # 7

print(5 and 6) # 6
print(5 & 6)   # 4

5
7
6
4


In [11]:
print(5 << 1)   # 10 
print(5 >> 1)  # 5

10
2
